# Arquitetura Bronze



In [0]:
from pyspark.sql import functions as F

CATALOGO = "atividade_2_cinedata"
spark.sql(f"USE CATALOG {CATALOGO}")
caminho = f"/Volumes/{CATALOGO}/raw/inputs/"

arquivos=[ 
    ("movies_info_TMDB_IMDB.csv", "bronze.tb_movies_info"),
    ("movies_financials_IMDB_TMDB.csv", "bronze.tb_movies_financials"),
    ("movies_metrics_IMDB_TMDB.csv", "bronze.tb_movies_metrics"),
    ("credits_and_tags_IMDB_TMDB.csv", "bronze.tb_credits_and_tags"),
    ("movies_reviews.csv", "bronze.tb_movies_reviews"),
    ]

In [0]:
def espiar(arquivo, n=5):
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(f"{caminho}{arquivo}"))
    print(f"{arquivo}  |  {len(df.columns)} colunas")
    display(df.limit(n))

espiar("movies_info_TMDB_IMDB.csv")
espiar("movies_financials_IMDB_TMDB.csv")
espiar("movies_metrics_IMDB_TMDB.csv")
espiar("credits_and_tags_IMDB_TMDB.csv")
espiar("movies_reviews.csv")

In [0]:
def ingerir_bronze(arquivo, tabela):
    df_raw = (spark.read
              .option("header", True)
              .option("multiline", True)
              .option("escape", '"')
              .csv(f"{caminho}{arquivo}"))
    
    # Auditoria
    df_bronze = df_raw.withColumn("ingestion_datetime", F.current_timestamp())

    # Escrita
    (df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela))
    
    print(f"{tabela}: {df_bronze.count()} linhas")

for arq, tab in arquivos:
    ingerir_bronze(arq, tab)

In [0]:
import socket
for host in ["olinda.bcb.gov.br", "api.github.com", "pypi.org"]:
    try:
        print(host, "->", socket.gethostbyname(host))
    except Exception as e:
        print(host, "-> FALHOU:", e)

In [0]:
import requests
import random
from datetime import datetime, timedelta, date

# Widgets: período da consulta, formato MM-DD-AAAA (padrão da API).
# Padrão: 30 dias corridos até hoje (a API não retorna fins de semana e feriados).
hoje = date.today()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=30)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")
data_inicial = datetime.strptime(data_inicio_formatada, "%m-%d-%Y").date()
data_final = datetime.strptime(data_fim_formatada, "%m-%d-%Y").date()

# URL no formato do enunciado. Montada à mão porque o Olinda exige '@' e '$' literais
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{data_inicial:%m-%d-%Y}'",
    "@dataFinalCotacao": f"'{data_final:%m-%d-%Y}'",
    "$format": "json",
    "$orderby": "dataHoraCotacao asc",  # Adicionado o '$'
    "$top": 1000,
    "$skip": 0
}

def gerar_mock(inicio, fim, cotacao_inicial=5.40, semente=42):
    """MOCK: simula a resposta da API PTAX com a MESMA estrutura do JSON original.
    Os valores são FICTÍCIOS e não representam a cotação oficial do BCB.
    Dias úteis recebem uma cotação com pequena variação; sábados e domingos ficam de
    fora, como na API real, para que o forward fill do Silver tenha lacunas a preencher."""
    random.seed(semente)               
    cotacao, registros, dia = cotacao_inicial, [], inicio
    while dia <= fim:
        if dia.weekday() < 5:          # 0-4 = segunda a sexta
            cotacao = round(cotacao + random.uniform(-0.05, 0.05), 4)
            registros.append({
                "dataHoraCotacao": f"{dia:%Y-%m-%d} 13:05:00.000",   # formato do Olinda
                "cotacaoCompra": cotacao,
            })
        dia += timedelta(days=1)
    return registros

# API real primeiro; mockup só se a rede estiver bloqueada ou a API falhar.
try:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    registros = resp.json()["value"]
    if not registros:
        raise ValueError("API retornou lista vazia para o período")
    fonte = "API BCB (dados reais)"
except (requests.exceptions.RequestException, ValueError) as e:
    # Free Edition bloqueia olinda.bcb.gov.br (DNS não resolve); o servidor também
    # pode responder 500. Em ambos os casos usamos o mockup.
    print(f"Falha na API: {type(e).__name__}")
    registros = gerar_mock(data_inicial, data_final)
    fonte = "MOCK (valores fictícios)"

print(f"FONTE DOS DADOS: {fonte} | {len(registros)} cotações")

df_cotacao = (
    spark.createDataFrame(
        [(r["dataHoraCotacao"], str(r["cotacaoCompra"])) for r in registros],
        schema="dataHoraCotacao string, cotacaoCompra string",
    )
    .withColumn("ingestion_datetime", F.current_timestamp())
)

df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
display(df_cotacao)